# Week 70, Day 1: Review: iteration

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w070d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def numbers():
    print("start")
    yield 1
    print("middle")
    yield 2

gen = numbers()
print("made")
print(next(gen))
print(list(gen))
print(list(gen))

In [ ]:
import itertools

def read_lines(text):
    for line in text.splitlines():
        yield line.strip()

def non_empty(lines):
    return (line for line in lines if line and not line.startswith("#"))

def parse(lines):
    for line in lines:
        name, value = line.split("=")
        yield name.strip(), int(value)

config = """
# settings
width = 80
height = 24

depth = 3
"""
pipeline = parse(non_empty(read_lines(config)))
print(next(pipeline))
print(dict(pipeline))
print(list(itertools.islice(itertools.count(10, 5), 4)), list(itertools.pairwise("abcd")))

In [ ]:
import functools

def counted(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

def repeat(times):
    def decorate(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return [func(*args, **kwargs) for _ in range(times)]
        return wrapper
    return decorate

@counted
@repeat(2)
def greet(name):
    "Say hello."
    return f"hi {name}"

print(greet("Ada"), greet.calls, greet.__name__, greet.__doc__)

def make_total():
    total = 0
    def add(x):
        nonlocal total
        total += x
        return total
    return add

running = make_total()
print(running(5), running(3), running.__closure__[0].cell_contents)

In [ ]:
from contextlib import contextmanager, ExitStack

log = []

@contextmanager
def opened(name):
    log.append(f"open {name}")
    try:
        yield name.upper()
    finally:
        log.append(f"close {name}")

try:
    with ExitStack() as stack:
        handles = [stack.enter_context(opened(n)) for n in ("a", "b", "c")]
        log.append(f"using {handles}")
        raise ValueError("halfway")
except ValueError:
    log.append("handled")
print(log)

> **Key idea.** Iterables give fresh iterators; iterators are single-use. Generators run lazily between yields, keep their state, and compose into pipelines that use constant memory; itertools supplies stages. Closures remember enclosing variables (nonlocal to rebind them); decorators replace functions, should use functools.wraps, and take arguments by returning a decorator. with always calls __exit__; contextmanager turns a one-yield generator into a context manager (yield inside try/finally); ExitStack manages a run-time number of them in reverse order. Watch for exhausted iterators, late binding, missing wraps and unprotected yields.

### Exercise: Sliding windows, lazily

Write a generator `windows(iterable, size)` yielding tuples of `size` consecutive items (`windows("abcd", 2)` gives `('a', 'b')`, `('b', 'c')`, `('c', 'd')`), keeping only `size` items in memory (use a `collections.deque` with `maxlen`), and working on infinite iterators. Then `moving_totals(iterable, size)`: a generator of the sums of each window.

In [ ]:
from collections import deque

def windows(iterable, size):
    ___

def moving_totals(iterable, size):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_windows():\n    \"lazy windows\"\n    import itertools, types\n    assert list(windows(\"abcd\", 2)) == [(\"a\", \"b\"), (\"b\", \"c\"), (\"c\", \"d\")] and list(windows([1, 2], 3)) == []\n    assert isinstance(windows([], 1), types.GeneratorType)\n    assert list(itertools.islice(windows(itertools.count(), 3), 2)) == [(0, 1, 2), (1, 2, 3)]\n    assert list(moving_totals([1, 2, 3, 4], 2)) == [3, 5, 7]")

### Exercise: A log pipeline

Write three generator stages and a function that uses them: `parse(lines)` yields dicts `{"level": ..., "message": ...}` from lines like `"ERROR disk full"` (the first word is the level; skip blank lines); `only(records, level)` yields the records with that level; and `first_n(records, n)` yields at most `n` records **and stops reading** after that. Then `first_errors(lines, n)`: the messages of the first `n` errors, as a list, without reading more lines than needed.

In [ ]:
def parse(lines):
    ___

def only(records, level):
    ___

def first_n(records, n):
    ___

def first_errors(lines, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pipeline():\n    \"a lazy pipeline\"\n    lines = [\"INFO start\", \"\", \"ERROR disk full\", \"WARN slow\", \"ERROR no network\", \"ERROR again\"]\n    assert list(parse(lines[:3])) == [{\"level\": \"INFO\", \"message\": \"start\"}, {\"level\": \"ERROR\", \"message\": \"disk full\"}]\n    assert first_errors(lines, 2) == [\"disk full\", \"no network\"] and first_errors(lines, 10)[-1] == \"again\"\n    read = []\n    def tracked():\n        for line in lines:\n            read.append(line)\n            yield line\n    first_errors(tracked(), 1)\n    assert read == lines[:3]\n    assert first_errors([], 3) == []")

### Exercise: A decorator with arguments

Write `retry(times, exceptions=(Exception,))`: a decorator factory whose decorated function is called up to `times` times while it raises one of `exceptions`, returning the first successful result or re-raising the last exception. Other exceptions propagate at once. Keep the function's name and docstring, and give the wrapper an `attempts` attribute recording how many attempts the **last** call made.

In [ ]:
import functools

def retry(times, exceptions=(Exception,)):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_retry():\n    \"retrying\"\n    replies = iter([ConnectionError(), ConnectionError(), \"ok\"])\n    @retry(5, (ConnectionError,))\n    def fetch():\n        \"Fetch a page.\"\n        r = next(replies)\n        if isinstance(r, Exception):\n            raise r\n        return r\n    assert fetch() == \"ok\" and fetch.attempts == 3 and fetch.__name__ == \"fetch\" and fetch.__doc__ == \"Fetch a page.\"\n    @retry(2, (ConnectionError,))\n    def always():\n        raise ConnectionError(\"down\")\n    try:\n        always()\n        assert False\n    except ConnectionError:\n        assert always.attempts == 2\n    @retry(3, (ConnectionError,))\n    def wrong():\n        raise KeyError\n    try:\n        wrong()\n        assert False\n    except KeyError:\n        assert wrong.attempts == 1")

### Exercise: Closures with state

Write `make_tally()`: returns a pair of functions `(add, report)`: `add(item)` counts one more of `item`; `report()` returns a dict of counts (a copy). Use a closure (no classes, no globals). Then `make_limiter(limit)`: returns a function `allow()` that returns `True` for its first `limit` calls and `False` afterwards, using `nonlocal`.

In [ ]:
def make_tally():
    ___

def make_limiter(limit):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_closures():\n    \"closures and nonlocal\"\n    add, report = make_tally()\n    for item in [\"tea\", \"jam\", \"tea\"]:\n        add(item)\n    r = report()\n    assert r == {\"tea\": 2, \"jam\": 1}\n    r[\"tea\"] = 99\n    assert report()[\"tea\"] == 2\n    other_add, other_report = make_tally()\n    assert other_report() == {}\n    allow = make_limiter(2)\n    assert [allow() for _ in range(4)] == [True, True, False, False]\n    assert \"nonlocal\" in source and \"global \" not in source and \"class \" not in source")

### Exercise: Context managers that clean up

Write a context manager class `Recorder(log, name)` that appends `"enter name"` to `log` on entry (returning itself) and `"exit name"` on exit, or `"exit name after ValueError"` (the exception's type name) if the block raised, never suppressing the exception. Then a generator-based context manager `override(settings, **changes)` (with `@contextmanager`) that applies the changes to the dict for the block and restores the old values afterwards, deleting keys that weren't there before, even after an error. Finally `enter_all(log, names)`: use one `ExitStack` to enter a `Recorder` for each name, and after the `with` block ends, return the names in the order they **exited**.

In [ ]:
from contextlib import contextmanager, ExitStack

class Recorder:
    def __init__(self, log, name):
        ___

@contextmanager
def override(settings, **changes):
    ___

def enter_all(log, names):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_managers():\n    \"with, contextmanager and ExitStack\"\n    log = []\n    with Recorder(log, \"a\") as r:\n        assert r.name == \"a\"\n    try:\n        with Recorder(log, \"b\"):\n            raise ValueError\n    except ValueError:\n        pass\n    assert log == [\"enter a\", \"exit a\", \"enter b\", \"exit b after ValueError\"]\n    settings = {\"mode\": \"slow\", \"level\": 1}\n    with override(settings, mode=\"fast\", debug=True):\n        assert settings == {\"mode\": \"fast\", \"level\": 1, \"debug\": True}\n    assert settings == {\"mode\": \"slow\", \"level\": 1}\n    try:\n        with override(settings, level=9):\n            raise KeyError\n    except KeyError:\n        pass\n    assert settings[\"level\"] == 1\n    assert enter_all([], [\"x\", \"y\", \"z\"]) == [\"z\", \"y\", \"x\"]")

### Bug hunt: A generator used twice

`squares` is a generator, so the first `sum` uses up every value and the `max` sees nothing, raising `ValueError`. Make it a list (or compute both from one pass) so this prints `14 9`.

In [ ]:
squares = (n * n for n in [1, 2, 3])
print(sum(squares), max(squares))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"single-use iterators\"\n    expect_output(\"14 9\")")

### Bug hunt: Every multiplier is the same

The lambdas made in the loop all look up `n` when they're **called**, by which time the loop has finished and `n` is 3, so this prints `[30, 30, 30]`. Bind each value when its lambda is made so this prints `[10, 20, 30]`.

In [ ]:
multipliers = [lambda x: x * n for n in (1, 2, 3)]
print([m(10) for m in multipliers])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"late binding\"\n    expect_output(\"[10, 20, 30]\")")

### Bug hunt: Clean-up that was skipped

The `@contextmanager` generator puts its clean-up after the `yield` without `try`/`finally`, so the exception raised in the block skips it and the log never says `released`: this prints `['acquired']`. Protect the clean-up so it prints `['acquired', 'released']`.

In [ ]:
from contextlib import contextmanager

log = []

@contextmanager
def lease():
    log.append("acquired")
    yield
    log.append("released")

try:
    with lease():
        raise RuntimeError("boom")
except RuntimeError:
    pass
print(log)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"try/finally around yield\"\n    expect_output(\"['acquired', 'released']\")")

### Puzzle: Flattening with yield from

Arrange the lines of a generator that flattens nested lists to any depth. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # list(zip(*[(1, 2), (3, 4)]))
q = ___   # g = (x for x in [1, 2, 3]); next(g); list(g)
r = ___   # True or False: iter(some_list) is some_list
s = ___   # what @functools.wraps copies onto the wrapper: "name and docstring" or "the arguments"
t = ___   # True or False: __exit__ is called when the with block raises

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [(1, 3), (2, 4)] and q == [2, 3] and r is False, \"p, q and r\"\n    assert s == \"name and docstring\" and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an instrumented pipeline

Combine the three tools. Write a decorator `stage(func)` for generator functions that take an iterable as their first argument: the decorated function returns a generator that passes items through `func`'s generator while counting, in the wrapper's attributes `taken` and `given`, how many items the stage read from its input and how many it yielded, **for the most recent run**. Keep the function's name. Then a context manager `pipeline(source, *stages, log=None)` (with `@contextmanager`) that chains the stages over the source (each stage's output is the next one's input) and yields the final generator; when the block ends, even after an error, it closes the final generator (so unfinished stages clean up) and, if `log` is given, appends `f"{name}: took {taken}, gave {given}"` for each stage, in order.

In [ ]:
import functools
from contextlib import contextmanager

def stage(func):
    ___

@contextmanager
def pipeline(source, *stages, log=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_instrumented():\n    \"decorators, generators and context managers together\"\n    @stage\n    def evens(items):\n        for x in items:\n            if x % 2 == 0:\n                yield x\n    @stage\n    def squared(items):\n        for x in items:\n            yield x * x\n    log = []\n    with pipeline(range(10), evens, squared, log=log) as out:\n        assert list(out) == [0, 4, 16, 36, 64]\n    assert log == [\"evens: took 10, gave 5\", \"squared: took 5, gave 5\"] and evens.__name__ == \"evens\"\n    log2 = []\n    import itertools\n    with pipeline(itertools.count(), evens, squared, log=log2) as out:\n        first = [next(out) for _ in range(3)]\n    assert first == [0, 4, 16] and log2 == [\"evens: took 5, gave 3\", \"squared: took 3, gave 3\"]\n    cleaned = []\n    @stage\n    def careful(items):\n        try:\n            for x in items:\n                yield x\n        finally:\n            cleaned.append(\"closed\")\n    try:\n        with pipeline(range(5), careful) as out:\n            next(out)\n            raise ValueError\n    except ValueError:\n        pass\n    assert cleaned == [\"closed\"]")

## How to review

For each section: read the summary, predict the example's output **before** running it, and only then check. Then do the exercises without looking back at the weeks they come from; when you get stuck, note the topic, look it up, and come back. The notes you make are your revision list for Friday.

Nothing runs until the first `next`; the generator pauses at each `yield`; and once it's exhausted, it stays empty. If any of that surprised you, start with Week 59.

## Iterators and generators

- The **iteration protocol** (Week 59): `iter(obj)` calls `__iter__` and returns an iterator; `next(it)` calls `__next__`, which raises `StopIteration` when there's nothing more. `for` loops, unpacking, `sum`, `list` and comprehensions all use it.
- An **iterable** can be iterated many times (each `iter()` gives a fresh iterator); an **iterator** is single-use and returns itself from `__iter__`.
- A **generator function** (one containing `yield`) returns a generator: a lazy iterator whose body runs only as values are asked for, keeping its local variables between steps. `yield from sub` hands over to another iterable.
- **Generator expressions** `(x * x for x in data)` are lazy comprehensions.
- **Pipelines** of generators process one item at a time: memory stays constant however big the input, and work stops as soon as the consumer stops asking. `itertools` (`islice`, `chain`, `groupby`, `takewhile`, `pairwise`, `batched`) provides ready-made stages.
- `gen.send(value)`, `gen.throw(exc)` and `gen.close()` drive generators from outside (and underlie Week 67's coroutines).

## Closures and decorators

- A **closure** (Week 60) is a nested function that remembers variables from the function around it, even after that function has returned. To **rebind** such a variable, declare it `nonlocal`.
- A **decorator** is a function that takes a function and returns a replacement, applied with `@`: `@timed` means `func = timed(func)`. Use **`functools.wraps`** so the replacement keeps the original's name, docstring and signature information.
- A decorator **with arguments** is a function that returns a decorator: `@retry(3)` means `func = retry(3)(func)`.
- Decorators can keep state (counts, caches) in the closure or as attributes on the wrapper.
- `functools` (Week 60 Day 4): `cache`, `lru_cache`, `partial`, `reduce`, `total_ordering`, `singledispatch`.

## Context managers

- **`with`** (Week 61) calls `__enter__` at the start (its result is what `as` binds) and **always** calls `__exit__(exc_type, exc, tb)` at the end, even after an error or a `return`. A true return value from `__exit__` suppresses the exception.
- **`@contextlib.contextmanager`** turns a generator with **one** `yield` into a context manager: code before the `yield` is the set-up, after it the clean-up. Put the `yield` in `try`/`finally`, or an error in the block skips the clean-up.
- **`contextlib`** also has `suppress`, `redirect_stdout`, `closing`, `nullcontext` and **`ExitStack`**, which manages any number of context managers and callbacks decided at run time, cleaning up in reverse order.
- Real uses: files, locks, temporary changes (directories, settings, attributes), transactions that roll back on error.

## Classic mistakes

- **Iterating an iterator twice**: the second pass is empty. Make a list, or call the generator function again.
- **Late binding**: closures made in a loop all see the loop variable's **final** value. Bind it as a default argument or make each closure in its own function call.
- **Forgetting `functools.wraps`**: the wrapper's name and docstring replace the original's, confusing debugging and tools.
- **A `@contextmanager` yield without `try`/`finally`**: an exception in the block skips the clean-up.
- **Modifying a collection while iterating over it**, and **calling a generator function but never iterating** (nothing happens).

## Practice

## Quiz

## Daily challenge

## Before you go

Write down the three ideas from this block you were least sure of. Tomorrow: the second review block, the memory model, attribute magic and CPython's internals.